# gridgen-c Example Comparison

Plots the reference grids from the
[gridgen-c examples directory](https://github.com/sakov/gridgen-c/tree/master/gridgen/examples)
(black lines) against grids regenerated by the current `cgrid_tools`
CRDT solver (red lines), using the same boundary definition and grid shape.

| Example | Description |
|---------|-------------|
| 1 | Estuary with one channel — primary CRDT paper figure |
| 2 | Same boundary as 1, turning points at different locations |
| 3 | Shape from the CRDT paper (wider, lower-orthogonality gap) |
| 4 | Non-right-angle corners (β = 4/3) on a circular arc boundary |

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from cgrid_tools.gridgen import BoundaryDefinition, CRDTOptions, GridGenerator, SolverMode

FIXTURES = Path("../tests/fixtures/gridgen_c")

In [ ]:
def parse_xy(path: Path):
    xs, ys, betas = [], [], []
    start_index = None
    for raw in path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        token = line.split()
        if len(token) < 2:
            continue
        x, y, beta, star = float(token[0]), float(token[1]), 0.0, False
        if len(token) >= 3:
            marker = token[2]
            if marker.endswith("*"):
                star = True
                marker = marker[:-1]
            if marker:
                beta = float(marker)
        xs.append(x)
        ys.append(y)
        betas.append(beta)
        if star:
            start_index = len(xs) - 1
    if start_index is None:
        start_index = 0
    return np.array(xs), np.array(ys), np.array(betas), start_index


def parse_prm(path: Path) -> dict:
    params = {}
    for line in path.read_text().splitlines():
        parts = line.split(None, 1)
        if len(parts) == 2:
            params[parts[0]] = parts[1].strip()
    return params


def parse_grid(path: Path, shape: tuple) -> tuple:
    ny, nx = shape
    body = path.read_text().splitlines()[1:]
    xs = np.full(ny * nx, np.nan)
    ys = np.full(ny * nx, np.nan)
    for i, line in enumerate(body):
        t = line.split()
        if t and t[0] != "NaN":
            xs[i] = float(t[0])
            ys[i] = float(t[1])
    return xs.reshape(ny, nx), ys.reshape(ny, nx)


def boundary_from_xy(path: Path) -> BoundaryDefinition:
    x, y, beta, start = parse_xy(path)
    x = np.roll(x, -start)
    y = np.roll(y, -start)
    beta = np.roll(beta, -start)
    return BoundaryDefinition(x=x, y=y, beta=beta)


def generate_grid(example_id: str):
    prm = parse_prm(FIXTURES / f"prm.{example_id}")
    nx = int(prm["nx"])
    ny = int(prm["ny"])
    shape = (ny, nx)
    options = CRDTOptions(
        nnodes=int(prm.get("nnodes", "10")),
        precision=float(prm.get("precision", "1.0e-12")),
        newton=bool(int(prm.get("newton", "1"))),
    )
    boundary = boundary_from_xy(FIXTURES / f"xy.{example_id}")
    result = GridGenerator(
        mode=SolverMode.GRIDGEN_CRDT, crdt_options=options
    ).generate(boundary, shape=shape)
    ref_x, ref_y = parse_grid(FIXTURES / f"grid.{example_id}", shape)
    return result.x, result.y, ref_x, ref_y, shape, boundary


def ortho_summary(x, y):
    from cgrid_tools import orthogonality_error
    err = orthogonality_error(x, y)
    finite = err[np.isfinite(err)]
    if finite.size == 0:
        return dict(mean=float("nan"), max=float("nan"), rms=float("nan"))
    return dict(
        mean=float(np.degrees(np.mean(np.abs(finite)))),
        max=float(np.degrees(np.max(np.abs(finite)))),
        rms=float(np.degrees(np.sqrt(np.mean(finite**2)))),
    )

In [ ]:
def plot_grid(ax, x, y, color, label, lw=0.4, alpha=1.0):
    """Draw all grid lines (rows then columns) on ax, masking NaN cells."""
    plotted = False
    for i in range(x.shape[0]):
        row_x = x[i, :]
        row_y = y[i, :]
        finite = np.isfinite(row_x) & np.isfinite(row_y)
        if not np.any(finite):
            continue
        # split at NaN gaps so lines don't bridge masked regions
        segments_x = np.where(finite, row_x, np.nan)
        segments_y = np.where(finite, row_y, np.nan)
        kw = dict(color=color, linewidth=lw, alpha=alpha)
        if not plotted:
            kw["label"] = label
            plotted = True
        ax.plot(segments_x, segments_y, **kw)
    for j in range(x.shape[1]):
        col_x = x[:, j]
        col_y = y[:, j]
        finite = np.isfinite(col_x) & np.isfinite(col_y)
        if not np.any(finite):
            continue
        segments_x = np.where(finite, col_x, np.nan)
        segments_y = np.where(finite, col_y, np.nan)
        ax.plot(segments_x, segments_y, color=color, linewidth=lw, alpha=alpha)


def plot_boundary(ax, boundary, color="navy", lw=1.0):
    ax.plot(boundary.x, boundary.y, color=color, linewidth=lw, linestyle="--",
            zorder=5, label="boundary")

## Example 1 — Estuary with one channel

40 × 40 grid. The boundary is a channel with a curved inlet. Turning
points are placed near the throat of the inlet (`β = −1` at vertex 7
and vertex 32 of the original xy file).

In [ ]:
our_x1, our_y1, ref_x1, ref_y1, shape1, bnd1 = generate_grid("1")

ref_o1 = ortho_summary(ref_x1, ref_y1)
our_o1 = ortho_summary(our_x1, our_y1)
print(f"Example 1 — shape {shape1}")
print(f"  gridgen-c : mean {ref_o1['mean']:.3f}°  max {ref_o1['max']:.3f}°  rms {ref_o1['rms']:.3f}°")
print(f"  ours      : mean {our_o1['mean']:.3f}°  max {our_o1['max']:.3f}°  rms {our_o1['rms']:.3f}°")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig.suptitle("Example 1 — Estuary (single channel), 40 × 40", fontsize=13)

for ax, gx, gy, color, title, label in [
    (axes[0], ref_x1, ref_y1, "black", "gridgen-c reference", "gridgen-c"),
    (axes[1], our_x1, our_y1, "red",   "cgrid_tools",   "cgrid_tools"),
]:
    plot_grid(ax, gx, gy, color=color, label=label)
    plot_boundary(ax, bnd1)
    o = ortho_summary(gx, gy)
    ax.set_title(f"{title}\nmean {o['mean']:.3f}°  max {o['max']:.3f}°", fontsize=10)
    ax.set_aspect("equal")
    ax.legend(fontsize=8, loc="upper left")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.show()

### Example 1 — overlaid comparison

Both grids on a single axes so positional differences are immediately visible.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
fig.suptitle("Example 1 — Estuary (single channel), overlaid", fontsize=13)

plot_grid(ax, ref_x1, ref_y1, color="black", label="gridgen-c", lw=0.5)
plot_grid(ax, our_x1, our_y1, color="red",   label="cgrid_tools", lw=0.5)
plot_boundary(ax, bnd1)

ax.set_aspect("equal")
ax.legend(fontsize=9)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

## Example 2 — Estuary with moved turning points

40 × 40 grid. Same boundary polygon as Example 1, but the `β = −1`
turning-point vertices are positioned further along the throat. This
produces a different, coarser grid despite an identical boundary shape —
demonstrating how turning-point placement implicitly controls cell density.

In [ ]:
our_x2, our_y2, ref_x2, ref_y2, shape2, bnd2 = generate_grid("2")

ref_o2 = ortho_summary(ref_x2, ref_y2)
our_o2 = ortho_summary(our_x2, our_y2)
print(f"Example 2 — shape {shape2}")
print(f"  gridgen-c : mean {ref_o2['mean']:.3f}°  max {ref_o2['max']:.3f}°  rms {ref_o2['rms']:.3f}°")
print(f"  ours      : mean {our_o2['mean']:.3f}°  max {our_o2['max']:.3f}°  rms {our_o2['rms']:.3f}°")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
fig.suptitle("Example 2 — Estuary (moved turning points), 40 × 40", fontsize=13)

for ax, gx, gy, color, title, label in [
    (axes[0], ref_x2, ref_y2, "black", "gridgen-c reference", "gridgen-c"),
    (axes[1], our_x2, our_y2, "red",   "cgrid_tools",   "cgrid_tools"),
]:
    plot_grid(ax, gx, gy, color=color, label=label)
    plot_boundary(ax, bnd2)
    o = ortho_summary(gx, gy)
    ax.set_title(f"{title}\nmean {o['mean']:.3f}°  max {o['max']:.3f}°", fontsize=10)
    ax.set_aspect("equal")
    ax.legend(fontsize=8, loc="upper left")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), constrained_layout=True)
fig.suptitle("Example 2 — Estuary (moved turning points), overlaid", fontsize=13)

plot_grid(ax, ref_x2, ref_y2, color="black", label="gridgen-c", lw=0.5)
plot_grid(ax, our_x2, our_y2, color="red",   label="cgrid_tools", lw=0.5)
plot_boundary(ax, bnd2)

ax.set_aspect("equal")
ax.legend(fontsize=9)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

## Example 3 — Shape from the CRDT paper

40 × 200 grid. A wider, more complex polygon with four convex (`β = 1`)
corners and several boundary waypoints. This example exposes a known
parity gap: our current `LogicalSides` mapper assigns different logical
corners than gridgen-c, leading to higher orthogonality error
(≈ 4° mean / 90° max vs gridgen-c's ≈ 0.9° mean / 40° max). Tracked as
Phase 7 item 6 in AGENTS.md.

In [ ]:
our_x3, our_y3, ref_x3, ref_y3, shape3, bnd3 = generate_grid("3")

ref_o3 = ortho_summary(ref_x3, ref_y3)
our_o3 = ortho_summary(our_x3, our_y3)
print(f"Example 3 — shape {shape3}")
print(f"  gridgen-c : mean {ref_o3['mean']:.3f}°  max {ref_o3['max']:.3f}°  rms {ref_o3['rms']:.3f}°")
print(f"  ours      : mean {our_o3['mean']:.3f}°  max {our_o3['max']:.3f}°  rms {our_o3['rms']:.3f}°")

fig, axes = plt.subplots(1, 2, figsize=(18, 5), constrained_layout=True)
fig.suptitle("Example 3 — CRDT paper shape, 40 × 200", fontsize=13)

for ax, gx, gy, color, title, label in [
    (axes[0], ref_x3, ref_y3, "black", "gridgen-c reference", "gridgen-c"),
    (axes[1], our_x3, our_y3, "red",   "cgrid_tools",   "cgrid_tools"),
]:
    plot_grid(ax, gx, gy, color=color, label=label)
    plot_boundary(ax, bnd3)
    o = ortho_summary(gx, gy)
    ax.set_title(f"{title}\nmean {o['mean']:.3f}°  max {o['max']:.3f}°", fontsize=10)
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5), constrained_layout=True)
fig.suptitle("Example 3 — CRDT paper shape, overlaid", fontsize=13)

plot_grid(ax, ref_x3, ref_y3, color="black", label="gridgen-c", lw=0.3)
plot_grid(ax, our_x3, our_y3, color="red",   label="cgrid_tools", lw=0.3)
plot_boundary(ax, bnd3)

ax.set_aspect("equal")
ax.legend(fontsize=9)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

## Example 4 — Circle with non-right-angle corners

40 × 40 grid. The boundary follows a circular arc with three corners
each carrying `β = 4/3` (non-right-angle — the interior angle is
120° rather than 90°). This is the gridgen-c showcase for the
non-right-angle corner capability. Agreement with gridgen-c is very
close (vertex mean ≈ 1.3 × 10⁻⁵).

In [ ]:
our_x4, our_y4, ref_x4, ref_y4, shape4, bnd4 = generate_grid("4")

ref_o4 = ortho_summary(ref_x4, ref_y4)
our_o4 = ortho_summary(our_x4, our_y4)
print(f"Example 4 — shape {shape4}")
print(f"  gridgen-c : mean {ref_o4['mean']:.3f}°  max {ref_o4['max']:.3f}°  rms {ref_o4['rms']:.3f}°")
print(f"  ours      : mean {our_o4['mean']:.3f}°  max {our_o4['max']:.3f}°  rms {our_o4['rms']:.3f}°")

fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)
fig.suptitle("Example 4 — Circle with non-right-angle corners (β = 4/3), 40 × 40", fontsize=13)

for ax, gx, gy, color, title, label in [
    (axes[0], ref_x4, ref_y4, "black", "gridgen-c reference", "gridgen-c"),
    (axes[1], our_x4, our_y4, "red",   "cgrid_tools",   "cgrid_tools"),
]:
    plot_grid(ax, gx, gy, color=color, label=label)
    plot_boundary(ax, bnd4)
    o = ortho_summary(gx, gy)
    ax.set_title(f"{title}\nmean {o['mean']:.3f}°  max {o['max']:.3f}°", fontsize=10)
    ax.set_aspect("equal")
    ax.legend(fontsize=8)
    ax.set_xlabel("x")
    ax.set_ylabel("y")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8), constrained_layout=True)
fig.suptitle("Example 4 — Circle (non-right-angle corners), overlaid", fontsize=13)

plot_grid(ax, ref_x4, ref_y4, color="black", label="gridgen-c", lw=0.5)
plot_grid(ax, our_x4, our_y4, color="red",   label="cgrid_tools", lw=0.5)
plot_boundary(ax, bnd4)

ax.set_aspect("equal")
ax.legend(fontsize=9)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

## Summary table — orthogonality error (deviation from 90°)

In [ ]:
rows = []
for ex_id, ox, oy, rx, ry, lbl in [
    ("1", our_x1, our_y1, ref_x1, ref_y1, "Estuary (single channel)"),
    ("2", our_x2, our_y2, ref_x2, ref_y2, "Estuary (moved turning pts)"),
    ("3", our_x3, our_y3, ref_x3, ref_y3, "CRDT paper shape"),
    ("4", our_x4, our_y4, ref_x4, ref_y4, "Circle (β = 4/3 corners)"),
]:
    ro = ortho_summary(rx, ry)
    oo = ortho_summary(ox, oy)
    rows.append((ex_id, lbl, ro, oo))

header = f"{'Ex':>3}  {'Description':<30}  {'ref mean':>9}  {'ref max':>9}  {'ours mean':>10}  {'ours max':>9}"
print(header)
print("-" * len(header))
for ex_id, lbl, ro, oo in rows:
    print(
        f"{ex_id:>3}  {lbl:<30}  {ro['mean']:9.3f}°  {ro['max']:9.3f}°  "
        f"{oo['mean']:10.3f}°  {oo['max']:9.3f}°"
    )

## Grid provenance and netCDF round-trip

Demonstrates the `options` dict attached to every `Grid` and the
`to_netcdf` / `Grid.from_netcdf` lossless round-trip.

In [ ]:
import os
import pprint
import tempfile

from cgrid_tools import Grid

# Re-generate example 1 briefly to show options
bnd1_demo = boundary_from_xy(FIXTURES / "xy.1")
prm1_demo = parse_prm(FIXTURES / "prm.1")
g_demo = GridGenerator(
    mode=SolverMode.GRIDGEN_CRDT,
    crdt_options=CRDTOptions(nnodes=10, precision=1e-12, newton=True)
).generate(bnd1_demo, shape=(int(prm1_demo["ny"]), int(prm1_demo["nx"])))
pprint.pprint(g_demo.options)
with tempfile.NamedTemporaryFile(suffix=".nc", delete=False) as f:
    path = f.name
g_demo.to_netcdf(path)
g_loaded = Grid.from_netcdf(path)
os.remove(path)
print(f"\nRound-trip: shape {g_demo.x.shape} → netCDF → shape {g_loaded.x.shape}")
assert g_demo.options == g_loaded.options
print("roundtrip ok")